# Die Suche von gestern messen

Gestern lieferte die Suche zu jeder Frage fünf Treffer. Ob sie taugt, zeigt eine Messung über Testfragen mit bekannten Fundstellen. AstroPlanner hat dafür 20 deutsche Fragen gesammelt, zu jeder die Abschnitte, die sie beantworten. Im Java-Lauf ergaben sie Hit Rate@5 0,95 und MRR@5 0,90. Das rechnen wir hier selbst nach.

### Aufgaben

**Für alle** (rund 12 Minuten, dazu die Ladezeit)

1. Testfragen ansehen und ablesen, welche Frage die Suche verfehlt *(Einstieg)*
2. `hit_rate` ergänzen, eine Zelle prüft sie
3. `mrr` ergänzen, dann mit dem Java-Lauf vergleichen und ohne Präfixe messen

**Bonus**

4. Eine Testfrage schreiben, deren Antwort im Korpus steht, die die Suche aber verfehlt
5. Knobelaufgabe: Wie gut trennt ein Schwellwert Fragen mit und ohne Antwort?

Zeitbox 15 Minuten, zu zweit. Am schnellsten mit *Laufzeit → Alle ausführen*, dann von oben lesen. Jede Aufgabe steht noch einmal an der Stelle, an der ihr sie macht.

## Für alle

> **Aufgabe 1** (Einstieg): *Laufzeit → Alle ausführen*, dann Schritt 1 bis 3 lesen. In Schritt 3 ablesen: Bei welcher Frage ist keine Fundstelle unter den ersten fünf, und bei welchen steht die erste erst auf Rang 2?

### 1. Modell, Chunks und Testfragen laden

Beim ersten Mal lädt das Modell rund 490 MB von Hugging Face.

In [ ]:
# Nur in Colab: Hilfsdateien und Daten aus dem Repo holen. Lokal tut die Zelle nichts.
import os
import subprocess
import sys

if "google.colab" in sys.modules:
    repo = "/content/ki-workshop-handson"
    if not os.path.exists(repo):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/dprinz/ki-workshop-handson.git", repo], check=True)
    os.chdir(f"{repo}/08-suche-messen")

In [1]:
import time

import numpy as np

from helfer import einbetten, lade_chunks, lade_fragen, lade_index, lade_modell, zeige, zeige_abschnitte, zeige_dokumente

start = time.perf_counter()
lade_modell()
chunks = lade_chunks()
vektoren = lade_index()  # mit den Präfixen, wie AstroPlanner
fragen = lade_fragen()
fragen_mit = [f for f in fragen if f["fundstellen"]]
fragen_ohne = [f for f in fragen if not f["fundstellen"]]
print(f"{len(chunks)} Chunks, {len(fragen_mit)} Fragen mit Fundstellen, {len(fragen_ohne)} ohne, "
      f"geladen in {time.perf_counter() - start:.1f} s")

643 Chunks, 20 Fragen mit Fundstellen, 3 ohne, geladen in 3.6 s


Eine Testfrage besteht aus dem Text und den Fundstellen: Paare aus Dokument und Abschnitt, die die Frage beantworten. Die drei Fragen ohne Fundstelle beantwortet der Korpus nicht, wie die nach dem Alter des Universums.

In [2]:
for frage in fragen[:3] + fragen_ohne[:1]:
    print(frage["id"], frage["frage"])
    for dokument, abschnitt in sorted(frage["fundstellen"]):
        print(f"     {dokument} – {abschnitt}")

D-01 Wofür brauche ich einen OIII-Filter?
     astronomical-filter – Specific emission line filters
     doubly-ionized-oxygen – Ion
     hoo-sho-and-lrgb – Narrowband filters
D-02 Was ist der Unterschied zwischen Breitband- und Schmalbandfiltern?
     astronomical-filter – Broadband
     astronomical-filter – Narrowband
     hoo-sho-and-lrgb – Narrowband filters
D-03 Was bedeutet Pixel Scale?
     pixel-scale-and-sampling – Introduction
N-01 Wie alt ist das Universum?


### 2. Die Suche von gestern

Dieselbe Schleife wie gestern. `zeige` markiert Treffer, die eine Fundstelle sind, mit ✓.

In [3]:
def suche(frage, vektoren, thema=None, k=5):
    treffer = []
    for chunk, vektor in zip(chunks, vektoren):
        if thema is not None and chunk["thema"] != thema:
            continue
        kosinus = np.dot(frage, vektor)  # beide Vektoren haben Länge 1: Skalarprodukt = Kosinus
        treffer.append((1 - kosinus, chunk))  # Distanz wie in Java und pgvector: 0 = gleiche Richtung
    treffer.sort(key=lambda t: t[0])
    return treffer[:k]


beispiel = next(f for f in fragen if f["id"] == "D-16")
print(beispiel["frage"])
zeige(suche(einbetten("query: " + beispiel["frage"]), vektoren), beispiel["fundstellen"])

Wann beginnt die astronomische Dämmerung?
✓ 1  0.173  Twilight – Astronomical twilight
            Astronomical twilight occurs when the geometric center of the Sun is b …
  2  0.186  Twilight – Between day and night
            because after the equinoxes the Sun's declination is then less than 4° …
  3  0.196  Twilight – Between day and night
            astronomical dusk in the entire year is 48º33'43". The maximum latitud …
  4  0.200  Twilight – Civil twilight
            Civil twilight is the period of time for which the geometric center of …
  5  0.203  Twilight – Lasting from one day to the next
            night. Nautical twilight: between about 54°34' and 60°34' north or sou …


### 3. Rang der ersten Fundstelle pro Frage

Ein Treffer zählt, wenn Dokument und Abschnitt zu einer Fundstelle passen. Pro Frage interessiert nur der Rang der ersten Fundstelle unter den fünf Treffern; 0 heisst, keine ist dabei.

In [4]:
def rang(frage, treffer):
    for platz, (distanz, chunk) in enumerate(treffer, start=1):
        if (chunk["id"], chunk["abschnitt"]) in frage["fundstellen"]:
            return platz
    return 0


def raenge_messen(vektoren, praefix="query: "):
    frage_vektoren = einbetten([praefix + f["frage"] for f in fragen_mit])  # alle 20 Fragen auf einmal
    return [rang(f, suche(v, vektoren)) for f, v in zip(fragen_mit, frage_vektoren)]


start = time.perf_counter()
raenge = raenge_messen(vektoren)
print(f"20 Fragen in {(time.perf_counter() - start) * 1000:.0f} ms\n")
for frage, r in zip(fragen_mit, raenge):
    print(f"{frage['id']}  Rang {r}  {frage['frage']}")

20 Fragen in 40 ms

D-01  Rang 1  Wofür brauche ich einen OIII-Filter?
D-02  Rang 1  Was ist der Unterschied zwischen Breitband- und Schmalbandfiltern?
D-03  Rang 1  Was bedeutet Pixel Scale?
D-04  Rang 1  Worin unterscheiden sich HOO und SHO?
D-05  Rang 1  Warum sollte ich zwischen den Aufnahmen dithern?
D-06  Rang 1  Wozu dienen Flat-Frames?
D-07  Rang 1  Was ist ein Dark-Frame und wozu braucht man ihn?
D-08  Rang 0  Was ist Seeing und wie wirkt es sich auf meine Bilder aus?
D-09  Rang 1  Soll ich eine Monochrom- oder eine Farbkamera verwenden?
D-10  Rang 1  Was passiert bei Undersampling?
D-11  Rang 2  Wie lange sollte eine einzelne Belichtung sein?
D-12  Rang 2  Warum wird das Bild mit mehr Integrationszeit besser?
D-13  Rang 1  Was ist Lucky Imaging?
D-14  Rang 1  Was sagt die Bortle-Skala aus?
D-15  Rang 1  Was ist die Airmass?
D-16  Rang 1  Wann beginnt die astronomische Dämmerung?
D-17  Rang 1  Was ist ein H-Alpha-Filter?
D-18  Rang 1  Wie funktioniert ein Autoguider?
D-19  Ran

### 4. Hit Rate@5 und MRR@5 ausrechnen

- **Hit Rate@5**: Anteil der Fragen, bei denen eine Fundstelle unter den ersten fünf Treffern ist.
- **MRR@5** (Mean Reciprocal Rank): Mittel von 1 / Rang der ersten Fundstelle. Rang 1 zählt 1, Rang 2 zählt 0,5, nicht gefunden zählt 0.

Das Beispiel von der Folie: Fundstelle auf Rang 1, auf Rang 2 und gar nicht, also `[1, 2, 0]`. Hit Rate 2 von 3, MRR (1 + 0,5 + 0) / 3 = 0,5.

`pruefe` ruft eine Funktion mit Beispielen auf, die man von Hand nachrechnen kann, und vergleicht mit dem Sollwert, wie ein kleiner Unit-Test.

In [5]:
def pruefe(funktion, faelle):
    richtig = 0
    for beispiel, soll in faelle:
        ist = funktion(beispiel)
        ok = isinstance(ist, (int, float)) and abs(ist - soll) < 0.001
        richtig += ok
        anzeige = f"{ist:.3f}" if isinstance(ist, (int, float)) else repr(ist)
        print(f"{'✓' if ok else '✗'} {funktion.__name__}({beispiel}) = {anzeige}, soll {soll:.3f}")
    print(f"{richtig} von {len(faelle)} stimmen")

> **Aufgabe 2**: `hit_rate` ergänzen. Fertig, wenn die Prüfung „4 von 4 stimmen“ meldet. Die letzte Zeile rechnet dann die Hit Rate@5 über die 20 Testfragen aus Schritt 3.

In [6]:
def hit_rate(raenge):
    return 0.0  # ergänzen


faelle_hit_rate = [([1, 2, 0], 2 / 3), ([1, 1, 1, 1], 1.0), ([0, 0, 0], 0.0), ([3, 0, 5, 0], 0.5)]
pruefe(hit_rate, faelle_hit_rate)
print(f"\n20 Testfragen: Hit Rate@5 {hit_rate(raenge):.2f}, im Java-Lauf 0.95")

✗ hit_rate([1, 2, 0]) = 0.000, soll 0.667
✗ hit_rate([1, 1, 1, 1]) = 0.000, soll 1.000
✓ hit_rate([0, 0, 0]) = 0.000, soll 0.000
✗ hit_rate([3, 0, 5, 0]) = 0.000, soll 0.500
1 von 4 stimmen

20 Testfragen: Hit Rate@5 0.00, im Java-Lauf 0.95


<details><summary>Tipp 1</summary>

Gezählt werden die Fragen, deren Rang grösser als 0 ist, geteilt durch die Anzahl aller Fragen, `len(raenge)`.

</details>
<details><summary>Tipp 2</summary>

`sum(1 for r in raenge if r > 0)` zählt die Ränge grösser als 0, wie `stream().filter(r -> r > 0).count()` in Java. Fehlt noch die Division.

</details>

In [7]:
# @title Lösung (aufklappen)
def loesung_hit_rate(raenge):
    return sum(1 for r in raenge if r > 0) / len(raenge)


pruefe(loesung_hit_rate, faelle_hit_rate)
print(f"\n20 Testfragen: Hit Rate@5 {loesung_hit_rate(raenge):.2f}, im Java-Lauf 0.95")

✓ loesung_hit_rate([1, 2, 0]) = 0.667, soll 0.667
✓ loesung_hit_rate([1, 1, 1, 1]) = 1.000, soll 1.000
✓ loesung_hit_rate([0, 0, 0]) = 0.000, soll 0.000
✓ loesung_hit_rate([3, 0, 5, 0]) = 0.500, soll 0.500
4 von 4 stimmen

20 Testfragen: Hit Rate@5 0.95, im Java-Lauf 0.95


> **Aufgabe 3**: `mrr` ergänzen, bis die Prüfung „4 von 4 stimmen“ meldet. Dann Schritt 5 und 6 lesen, in Schritt 6 schätzen, bevor ihr die Ausgabe anseht. Fertig, wenn ihr eure Schätzung mit dem Ergebnis verglichen habt.

In [8]:
def mrr(raenge):
    return 0.0  # ergänzen


faelle_mrr = [([1, 2, 0], 0.5), ([1, 1, 1, 1], 1.0), ([0, 0, 0], 0.0), ([4, 0], 0.125)]
pruefe(mrr, faelle_mrr)
print(f"\n20 Testfragen: MRR@5 {mrr(raenge):.2f}, im Java-Lauf 0.90")

✗ mrr([1, 2, 0]) = 0.000, soll 0.500
✗ mrr([1, 1, 1, 1]) = 0.000, soll 1.000
✓ mrr([0, 0, 0]) = 0.000, soll 0.000
✗ mrr([4, 0]) = 0.000, soll 0.125
1 von 4 stimmen

20 Testfragen: MRR@5 0.00, im Java-Lauf 0.90


<details><summary>Tipp 1</summary>

Wie bei der Hit Rate, nur zählt eine gefundene Frage nicht 1, sondern 1 / Rang. Rang 0 heisst nicht gefunden und zählt 0; durch 0 teilen geht nicht, diese Ränge also auslassen.

</details>
<details><summary>Tipp 2</summary>

`sum(1 / r for r in raenge if r > 0)`, dann durch die Anzahl aller Fragen teilen, auch der nicht gefundenen. Sonst kommt bei `[4, 0]` 0,25 heraus statt 0,125.

</details>

In [9]:
# @title Lösung (aufklappen)
def loesung_mrr(raenge):
    # Rang 0 fällt aus der Summe, geteilt wird trotzdem durch alle Fragen.
    return sum(1 / r for r in raenge if r > 0) / len(raenge)


pruefe(loesung_mrr, faelle_mrr)
print(f"\n20 Testfragen: MRR@5 {loesung_mrr(raenge):.2f}, im Java-Lauf 0.90")

✓ loesung_mrr([1, 2, 0]) = 0.500, soll 0.500
✓ loesung_mrr([1, 1, 1, 1]) = 1.000, soll 1.000
✓ loesung_mrr([0, 0, 0]) = 0.000, soll 0.000
✓ loesung_mrr([4, 0]) = 0.125, soll 0.125
4 von 4 stimmen

20 Testfragen: MRR@5 0.90, im Java-Lauf 0.90


### 5. Vergleich mit dem Java-Lauf

Die Zelle rechnet mit den Funktionen aus den Lösungen, damit sie auch läuft, wenn Schritt 4 noch offen ist. Dazu der beste Abstand: für die Fragen mit Fundstelle und für die drei ohne.

In [10]:
def bester_abstand(liste):
    werte = [suche(einbetten("query: " + f["frage"]), vektoren)[0][0] for f in liste]
    return f"{min(werte):.3f} bis {max(werte):.3f}"


verfehlt = [f["id"] for f, r in zip(fragen_mit, raenge) if r == 0]
print(f"{'':30}{'Java':>17}{'hier':>17}")
print(f"{'Hit Rate@5':30}{'0.95':>17}{loesung_hit_rate(raenge):>17.2f}")
print(f"{'MRR@5':30}{'0.90':>17}{loesung_mrr(raenge):>17.2f}")
print(f"{'nicht unter den ersten fünf':30}{'D-08':>17}{', '.join(verfehlt):>17}")
print(f"{'bester Abstand mit Fundstelle':30}{'0.118 bis 0.205':>17}{bester_abstand(fragen_mit):>17}")
print(f"{'bester Abstand ohne':30}{'0.203 bis 0.229':>17}{bester_abstand(fragen_ohne):>17}")

                                           Java             hier
Hit Rate@5                                 0.95             0.95
MRR@5                                      0.90             0.90
nicht unter den ersten fünf                D-08             D-08
bester Abstand mit Fundstelle   0.118 bis 0.205  0.127 bis 0.208


bester Abstand ohne             0.203 bis 0.229  0.206 bis 0.230


Dieselben Kennzahlen wie in Java. Einzelne Ränge unterscheiden sich trotzdem: Java hat die Fundstelle bei D-11 und D-17 auf Rang 2, hier bei D-11 und D-12. Die Ursache ist dieselbe wie gestern: Java nutzt eine quantisierte Fassung des Modells, die Distanzen weichen in der dritten Stelle ab, und bei knappen Abständen tauschen zwei Treffer den Platz.

Auch die Abstände überlappen wie in Java: Die schlechteste Frage mit Fundstelle liegt weiter weg als die beste ohne. Ein fester Schwellwert trennt die beiden Gruppen deshalb nicht sauber. Wie viele Fragen er falsch einordnet, misst Aufgabe 5.

Die Frage, die die Suche verfehlt:

In [11]:
d08 = next(f for f in fragen if f["id"] == "D-08")
print(d08["frage"], "\nFundstellen:", sorted(d08["fundstellen"]), "\n")
zeige(suche(einbetten("query: " + d08["frage"]), vektoren), d08["fundstellen"])

Was ist Seeing und wie wirkt es sich auf meine Bilder aus? 
Fundstellen: [('astronomical-seeing', 'Effects'), ('astronomical-seeing', 'Introduction')] 

  1  0.207  Dithering – Introduction
            Dithering means moving the telescope by a small, random amount between …
  2  0.209  Image noise – Introduction
            Image noise is random variation of brightness or color information in …
  3  0.209  Bayer filter – Zippering artifact
            The zippering artifact is another side effect of CFA demosaicing, whic …
  4  0.212  Image noise – Noise reduction
            An image is a picture, photograph or any other form of 2D representati …
  5  0.216  F-number – Conventional scales
            The word stop is sometimes confusing due to its multiple meanings. A s …


### 6. Ohne Präfixe messen

Gestern in den Bonusaufgaben: Ohne `query: ` und `passage: ` ändern sich die Distanzen etwas. Was macht das mit den Kennzahlen? Erst schätzen, dann die Ausgabe ansehen.

In [12]:
vektoren_ohne = lade_index(praefix=False)
raenge_ohne = raenge_messen(vektoren_ohne, praefix="")

print(f"{'':16}{'Hit Rate@5':>12}{'MRR@5':>8}{'Java':>16}")
print(f"{'mit Präfixen':16}{loesung_hit_rate(raenge):>12.2f}{loesung_mrr(raenge):>8.2f}{'0.95 / 0.90':>16}")
print(f"{'ohne Präfixe':16}{loesung_hit_rate(raenge_ohne):>12.2f}{loesung_mrr(raenge_ohne):>8.2f}{'0.95 / 0.89':>16}")
print()
for frage, mit, ohne in zip(fragen_mit, raenge, raenge_ohne):
    if mit != ohne:
        print(f"{frage['id']}  Rang mit {mit}, ohne {ohne}  {frage['frage']}")

                  Hit Rate@5   MRR@5            Java
mit Präfixen            0.95    0.90     0.95 / 0.90
ohne Präfixe            0.95    0.93     0.95 / 0.89

D-12  Rang mit 2, ohne 1  Warum wird das Bild mit mehr Integrationszeit besser?


Hier wird die Suche ohne Präfixe etwas besser, in Java etwas schlechter (dort rutscht D-11 von Rang 2 auf Rang 3). Beides hängt an einer einzigen Frage: Bei 20 Fragen verschiebt ein Rang 2 statt 1 die MRR um 0,025. Ein Unterschied dieser Grösse ist mit 20 Fragen nicht von Zufall zu trennen. Für eine Entscheidung braucht es mehr Fragen, und zwar solche, an denen man nicht schon nachgebessert hat.

## Bonus: eine Frage, die die Suche verfehlt

> **Aufgabe 4** (Bonus): Schreibt eine Testfrage, deren Antwort im Korpus steht, die die Suche aber nicht unter die ersten fünf bringt. Zuerst die Fundstelle suchen: `zeige_dokumente` listet die Dokumente, `zeige_abschnitte` die Abschnitte eines Dokuments.

Ideen: ein umgangssprachliches Wort statt des Fachbegriffs, ein Symptom statt der Ursache, eine Abkürzung, zwei Fragen in einer.

In [13]:
zeige_dokumente(chunks)
print()
zeige_abschnitte(chunks, "astronomical-filter")

filters: astronomical-filter, bortle-scale, doubly-ionized-oxygen, false-color, hoo-sho-and-lrgb, hydrogen-alpha, light-pollution
imaging: air-mass, astronomical-seeing, astrophotography, calibration-frames, dark-frame-subtraction, dithering, extinction, flat-field-correction, image-noise, lucky-imaging, pixel-scale-and-sampling, shift-and-add, shot-noise, signal-to-noise-ratio-imaging, sub-exposures-and-integration, twilight
equipment: active-pixel-sensor, angular-resolution, apochromat, autoguider, bayer-filter, charge-coupled-device, equatorial-mount, f-number, field-of-view, focal-length, mono-and-osc-cameras, newtonian-telescope, refracting-telescope, schmidt-cassegrain-telescope, telescope
targets: apparent-magnitude, emission-nebula, galaxy, globular-cluster, h-ii-region, messier-object, open-cluster, planetary-nebula, supernova-remnant, surface-brightness

Introduction
          An astronomical filter is a telescope accessory consisting of an optic …
White light filters
       

In [14]:
meine_frage = {
    "frage": "Brauche ich bei Vollmond einen Filter?",
    "fundstellen": {("astronomical-filter", "Moon filters")},
}

treffer = suche(einbetten("query: " + meine_frage["frage"]), vektoren)
zeige(treffer, meine_frage["fundstellen"])
print("\nRang der ersten Fundstelle:", rang(meine_frage, treffer) or "nicht unter den ersten fünf")

✓ 1  0.199  Astronomical filter – Moon filters
            Neutral density filters, also known in astronomy as Moon filters, are …
  2  0.206  Bayer filter – Modifications
            The Bayer filter is almost universal on consumer digital cameras. Alte …
  3  0.206  Astrophotography – Filters
            Filters can be categorised into two classes; broadband and narrowband. …
  4  0.207  Astronomical filter – Color filters
            Color filters work by absorption/transmission, and can tell which part …
  5  0.209  HOO, SHO and LRGB – Narrowband filters
            Emission nebulae shine in a few spectral lines. The most important for …

Rang der ersten Fundstelle: 1


<details><summary>Tipp</summary>

Am ehesten klappt es, wenn die Frage den Fachbegriff nicht nennt. Ein Dokument wählen, dessen Titel ein Fachbegriff ist, zum Beispiel `astronomical-seeing`, `flat-field-correction` oder `shot-noise`, und nach dem fragen, was man am Himmel oder im Bild sieht.

</details>

In [15]:
# @title Lösung (aufklappen)
loesung_frage = {
    "frage": "Warum funkeln die Sterne?",
    "fundstellen": {("astronomical-seeing", "Introduction"), ("astronomical-seeing", "Effects")},
}
loesung_treffer = suche(einbetten("query: " + loesung_frage["frage"]), vektoren)
zeige(loesung_treffer, loesung_frage["fundstellen"])
print("\nRang der ersten Fundstelle:", rang(loesung_frage, loesung_treffer) or "nicht unter den ersten fünf")

  1  0.216  Planetary nebula – Central stars
            The central stars of planetary nebulae are very hot. Only when a star …
  2  0.219  Globular cluster – Introduction
            A globular cluster is a spheroidal conglomeration of stars that is bou …
  3  0.219  Galaxy – Future trends
            Spiral galaxies, like the Milky Way, produce new generations of stars …
  4  0.220  Galaxy – Evolution
            Once stars begin to form, emit radiation, and in some cases explode, t …
  5  0.221  Planetary nebula – Introduction
            (P.N.N.), ionizes the ejected material. Absorbed ultraviolet light the …

Rang der ersten Fundstelle: nicht unter den ersten fünf


Der Abschnitt zu Seeing nennt das Funkeln ausdrücklich („twinkling“), die Suche findet ihn trotzdem nicht. Ein weiteres Beispiel: „Mein Bild ist in den Ecken dunkler als in der Mitte, was tun?“ Gemeint ist Vignettierung, die Antwort steht bei den Flat-Frames.

In einem echten Projekt wird so ein Fehler ein neuer Fall im Fragenset, wie ein Regressionstest nach einem Bug. Danach ist zu klären, was hilft: ein anderer Suchtext (das Modell schreibt ihn in AstroPlanner selbst), ein anderes Embedding-Modell oder ein Dokument, das den Begriff erklärt.

## Bonus: Wie gut trennt ein Schwellwert?

Eine Anwendung soll bei einer Frage, deren Antwort nicht im Korpus steht, sagen, dass sie nichts findet. Die einfachste Regel: Liegt schon der beste Treffer weiter weg als ein Schwellwert, gilt die Frage als nicht beantwortet. Das Fragenset hat beide Arten, 20 Fragen mit Fundstelle und 3 ohne. Ein Schwellwert kann sich auf zwei Arten irren:

- **abgewiesen**: Der beste Treffer einer Frage mit Fundstelle liegt weiter weg als der Schwellwert.
- **durchgelassen**: Der beste Treffer einer Frage ohne Fundstelle liegt höchstens so weit weg wie der Schwellwert.

> **Aufgabe 5** (Bonus, Knobelaufgabe): `fehler(schwelle)` ergänzen, sodass sie beide Zahlen zurückgibt. Die Zelle prüft zwei Werte und druckt eine Tabelle von 0,17 bis 0,24. Welcher Schwellwert macht am wenigsten Fehler? Und was heisst das Ergebnis für eine Anwendung, die sich auf diesen Wert verlässt?

In [16]:
def beste_abstaende(liste):
    frage_vektoren = einbetten(["query: " + f["frage"] for f in liste])  # alle Fragen auf einmal
    return [suche(v, vektoren)[0][0] for v in frage_vektoren]  # Distanz des besten Treffers


abstand_mit = beste_abstaende(fragen_mit)
abstand_ohne = beste_abstaende(fragen_ohne)
print("mit Fundstelle: ", " ".join(f"{a:.3f}" for a in sorted(abstand_mit)))
print("ohne Fundstelle:", " ".join(f"{a:.3f}" for a in sorted(abstand_ohne)))

mit Fundstelle:  0.127 0.128 0.130 0.140 0.144 0.144 0.159 0.164 0.165 0.171 0.172 0.173 0.175 0.182 0.185 0.197 0.198 0.199 0.207 0.208
ohne Fundstelle: 0.206 0.224 0.230


In [17]:
def fehler(schwelle):
    abgewiesen = 0  # ergänzen: Fragen mit Fundstelle, deren bester Abstand grösser ist als die Schwelle
    durchgelassen = 0  # ergänzen: Fragen ohne Fundstelle, deren bester Abstand höchstens die Schwelle ist
    return abgewiesen, durchgelassen


def pruefe_schwellen(funktion):
    for schwelle, soll in [(0.20, (2, 0)), (0.22, (0, 1))]:
        ist = tuple(funktion(schwelle))
        print(f"{'✓' if ist == soll else '✗'} {funktion.__name__}({schwelle}) = {ist}, soll {soll}")
    print(f"\n{'Schwelle':>8}{'abgewiesen':>12}{'durchgelassen':>15}{'Fehler':>8}")
    for schwelle in [0.17, 0.18, 0.19, 0.20, 0.21, 0.22, 0.23, 0.24]:
        abgewiesen, durchgelassen = funktion(schwelle)
        print(f"{schwelle:>8.2f}{abgewiesen:>12}{durchgelassen:>15}{abgewiesen + durchgelassen:>8}")


pruefe_schwellen(fehler)

✗ fehler(0.2) = (0, 0), soll (2, 0)
✗ fehler(0.22) = (0, 0), soll (0, 1)

Schwelle  abgewiesen  durchgelassen  Fehler
    0.17           0              0       0
    0.18           0              0       0
    0.19           0              0       0
    0.20           0              0       0
    0.21           0              0       0
    0.22           0              0       0
    0.23           0              0       0
    0.24           0              0       0


<details><summary>Tipp 1</summary>

`abstand_mit` und `abstand_ohne` sind Listen mit dem besten Abstand pro Frage. Gezählt wird wie bei der Hit Rate: `sum(1 for a in … if …)`.

</details>
<details><summary>Tipp 2</summary>

`abgewiesen = sum(1 for a in abstand_mit if a > schwelle)`, `durchgelassen` entsprechend mit `abstand_ohne` und `<=`. Für die zweite Frage: Wie viele Fragen ohne Fundstelle entscheiden darüber, welcher Wert am besten ist?

</details>

In [18]:
# @title Lösung (aufklappen)
def fehler_loesung(schwelle):
    abgewiesen = sum(1 for a in abstand_mit if a > schwelle)
    durchgelassen = sum(1 for a in abstand_ohne if a <= schwelle)
    return abgewiesen, durchgelassen


pruefe_schwellen(fehler_loesung)

✓ fehler_loesung(0.2) = (2, 0), soll (2, 0)
✓ fehler_loesung(0.22) = (0, 1), soll (0, 1)

Schwelle  abgewiesen  durchgelassen  Fehler
    0.17          11              0      11
    0.18           7              0       7
    0.19           5              0       5
    0.20           2              0       2
    0.21           0              1       1
    0.22           0              1       1
    0.23           0              2       2
    0.24           0              3       3


Am wenigsten Fehler macht ein Schwellwert von 0,21 oder 0,22: Alle 20 Fragen mit Fundstelle kommen durch, aber auch die Frage nach dem Alter des Universums (0,206). Darunter werden schnell Fragen mit Fundstelle abgewiesen, bei 0,19 schon fünf; darüber kommen auch die beiden anderen Fragen ohne Fundstelle durch. Kein Wert trennt alle 23 Fragen.

Was das für eine Anwendung heisst:

- **Dünne Grundlage:** Über den besten Wert entscheiden drei Fragen ohne Fundstelle, genau genommen eine. Eine vierte, die bei 0,20 liegt, verschiebt ihn. Für einen Schwellwert braucht das Fragenset viele Fälle in beide Richtungen.
- **Am selben Set eingestellt:** Der Wert ist an diesen 23 Fragen ausgesucht. Ob er hält, zeigen erst Fragen, an denen nicht eingestellt wurde, wie das Testset beim Training.
- **Durchgelassen heisst nicht gefunden:** D-08 kommt bei 0,21 durch, aber keine ihrer Fundstellen ist unter den ersten fünf. Der Schwellwert prüft nur, ob etwas Nahes da ist.
- **Welcher Fehler schlimmer ist, hängt von der Anwendung ab:** Eine abgewiesene Frage endet mit „nichts gefunden“, obwohl die Antwort im Korpus steht. Eine durchgelassene gibt dem Sprachmodell fremde Chunks, aus denen es trotzdem eine Antwort bauen kann.

## Was ihr gesehen habt

Hit Rate@5 und MRR@5 sind je eine Zeile Python, und über die 20 Testfragen kommen dieselben Werte heraus wie im Java-Lauf, 0,95 und 0,90. Steht bei einer einzigen Frage die Fundstelle auf Rang 2 statt 1, sinkt die MRR um 0,025; Unterschiede dieser Grösse, wie mit und ohne Präfixe, sind mit 20 Fragen kein Befund. Ein Fragenset wächst wie eine Testsuite: Jede Frage, an der die Suche scheitert, wird ein neuer Fall.